In [7]:
print("Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64")

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Đọc dữ liệu Iris và chuẩn hóa về đoạn [0, 1]
iris = load_iris()
X = iris.data / np.max(iris.data, axis=0)
y = iris.target

print(f"Đã chuẩn bị dữ liệu: X shape = {X.shape}, y shape = {y.shape}")
print("Tên các loài hoa (classes):", iris.target_names)

Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64
Đã chuẩn bị dữ liệu: X shape = (150, 4), y shape = (150,)
Tên các loài hoa (classes): ['setosa' 'versicolor' 'virginica']


In [8]:
# CÂU c)
print("Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

def create_model():
    model = Sequential([
        # Dùng layer Input chuẩn thay vì truyền input_shape vào Dense để hết cảnh báo UserWarning
        Input(shape=(4,)),
        Dense(10, activation='relu'),
        Dense(20, activation='relu'),
        Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# In cấu trúc kiểm tra
demo = create_model()
demo.summary()

Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_33 (Dense)                │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# CÂU d)
print("Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64")

import warnings
import tensorflow as tf

# Tắt cảnh báo hiển thị giữa các fold
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = []
acc_per_fold = []
prec_macro_per_fold = []
rec_macro_per_fold = []
f1_macro_per_fold = []

prec_per_class = []
rec_per_class = []
f1_per_class = []

fold_no = 1
for train_idx, val_idx in kfold.split(X):
    model = create_model()

    X_train_f, X_val_f = X[train_idx], X[val_idx]
    y_train_f, y_val_f = y[train_idx], y[val_idx]

    # Huấn luyện mô hình
    model.fit(X_train_f, y_train_f, epochs=100, batch_size=2, verbose=0)

    # Dùng gọi trực tiếp mô hình model(X_val_f) để triệt tiêu TensorFlow retracing warning
    y_pred_prob = model(X_val_f, training=False).numpy()
    y_pred_val = np.argmax(y_pred_prob, axis=1)

    acc = accuracy_score(y_val_f, y_pred_val)
    prec = precision_score(y_val_f, y_pred_val, average='macro', zero_division=0)
    rec = recall_score(y_val_f, y_pred_val, average='macro', zero_division=0)
    f1 = f1_score(y_val_f, y_pred_val, average='macro', zero_division=0)

    prec_cls = precision_score(y_val_f, y_pred_val, average=None, zero_division=0, labels=[0, 1, 2])
    rec_cls = recall_score(y_val_f, y_pred_val, average=None, zero_division=0, labels=[0, 1, 2])
    f1_cls = f1_score(y_val_f, y_pred_val, average=None, zero_division=0, labels=[0, 1, 2])

    models.append(model)
    acc_per_fold.append(acc)
    prec_macro_per_fold.append(prec)
    rec_macro_per_fold.append(rec)
    f1_macro_per_fold.append(f1)

    prec_per_class.append(prec_cls)
    rec_per_class.append(rec_cls)
    f1_per_class.append(f1_cls)

    print(f"Fold {fold_no:2d}: Accuracy = {acc*100:6.2f}% | Precision = {prec:.4f} | Recall = {rec:.4f} | F1-score = {f1:.4f}")
    fold_no += 1

Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64
Fold  1: Accuracy =  93.33% | Precision = 0.9167 | Recall = 0.9444 | F1-score = 0.9221
Fold  2: Accuracy = 100.00% | Precision = 1.0000 | Recall = 1.0000 | F1-score = 1.0000
Fold  3: Accuracy = 100.00% | Precision = 1.0000 | Recall = 1.0000 | F1-score = 1.0000
Fold  4: Accuracy =  93.33% | Precision = 0.9524 | Recall = 0.9333 | F1-score = 0.9373
Fold  5: Accuracy = 100.00% | Precision = 1.0000 | Recall = 1.0000 | F1-score = 1.0000
Fold  6: Accuracy =  93.33% | Precision = 0.9167 | Recall = 0.9444 | F1-score = 0.9221
Fold  7: Accuracy =  86.67% | Precision = 0.9167 | Recall = 0.8333 | F1-score = 0.8413
Fold  8: Accuracy = 100.00% | Precision = 1.0000 | Recall = 1.0000 | F1-score = 1.0000
Fold  9: Accuracy = 100.00% | Precision = 1.0000 | Recall = 1.0000 | F1-score = 1.0000
Fold 10: Accuracy = 100.00% | Precision = 1.0000 | Recall = 1.0000 | F1-score = 1.0000


In [10]:
# CÂU e)
print("Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64")

avg_acc = np.mean(acc_per_fold)
avg_rec = np.mean(rec_macro_per_fold)
avg_prec = np.mean(prec_macro_per_fold)
avg_f1 = np.mean(f1_macro_per_fold)

print("--- HIỆU SUẤT TRUNG BÌNH CỦA MÔ HÌNH SAU 10-FOLD CROSS VALIDATION ---")
print(f"Average Accuracy:  {avg_acc * 100:.2f}%")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall:    {avg_rec:.4f}")
print(f"Average F1-score:  {avg_f1:.4f}")

Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64
--- HIỆU SUẤT TRUNG BÌNH CỦA MÔ HÌNH SAU 10-FOLD CROSS VALIDATION ---
Average Accuracy:  96.67%
Average Precision: 0.9702
Average Recall:    0.9656
Average F1-score:  0.9623


In [11]:
# CÂU f)
print("Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64")

avg_prec_classes = np.mean(prec_per_class, axis=0)
avg_rec_classes = np.mean(rec_per_class, axis=0)
avg_f1_classes = np.mean(f1_per_class, axis=0)

df_per_class = pd.DataFrame({
    'Class': [f"Class {i} ({name})" for i, name in enumerate(iris.target_names)],
    'Average Precision': avg_prec_classes,
    'Average Recall': avg_rec_classes,
    'Average F1-score': avg_f1_classes
})

print("--- KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH CHO TỪNG CLASS ---")
print(df_per_class.to_string(index=False))

Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64
--- KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH CHO TỪNG CLASS ---
               Class  Average Precision  Average Recall  Average F1-score
    Class 0 (setosa)           1.000000        1.000000          1.000000
Class 1 (versicolor)           0.985714        0.916667          0.940793
 Class 2 (virginica)           0.925000        0.980000          0.946032


In [12]:
# CÂU g)
print("Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64")

# Lựa chọn mô hình có F1-score cao nhất qua các fold
best_idx = int(np.argmax(f1_macro_per_fold))
best_model = models[best_idx]
print(f"Mô hình được chọn: Fold {best_idx + 1} (F1-score cao nhất = {f1_macro_per_fold[best_idx]:.4f})\n")

# 03 mẫu dữ liệu mới đề bài yêu cầu
new_samples = np.array([
    [6.2, 2.9, 4.3, 1.3],
    [5.1, 3.5, 1.4, 0.2],
    [7.3, 2.8, 6.4, 2.1]
])

# Chuẩn hóa 3 mẫu mới theo max từng cột ban đầu
new_samples_norm = new_samples / np.max(iris.data, axis=0)

# Dự đoán xác suất và nhãn phân lớp
preds_prob = best_model.predict(new_samples_norm, verbose=0)
preds_class = np.argmax(preds_prob, axis=1)

print("KẾT QUẢ DỰ ĐOÁN 03 MẪU MỚI:")
for i in range(len(new_samples)):
    label_id = preds_class[i]
    label_name = iris.target_names[label_id]
    confidence = preds_prob[i][label_id] * 100
    print(f"Mẫu {i+1} {new_samples[i]} => Dự đoán: Lớp {label_id} ({label_name}) với độ tin cậy {confidence:.2f}%")

Họ và tên: Trần Thanh Ngân - MSSV: 6451071048 - Lớp: CNTT K64
Mô hình được chọn: Fold 2 (F1-score cao nhất = 1.0000)

KẾT QUẢ DỰ ĐOÁN 03 MẪU MỚI:
Mẫu 1 [6.2 2.9 4.3 1.3] => Dự đoán: Lớp 1 (versicolor) với độ tin cậy 99.85%
Mẫu 2 [5.1 3.5 1.4 0.2] => Dự đoán: Lớp 0 (setosa) với độ tin cậy 99.99%
Mẫu 3 [7.3 2.8 6.4 2.1] => Dự đoán: Lớp 2 (virginica) với độ tin cậy 99.90%
